# Project 05: a parallel N-1 contingency screening toolkit

**Tools:** pandapower, Python, concurrent.futures
**Network:** IEEE 118-bus (Christie, 1993, University of Washington archive) via pandapower
**Notebook status:** executed on a 4 core machine, all numbers below are measured

---

## The question

> N-1 screening is embarrassingly parallel: every contingency is independent. How much
> speed-up does that actually buy, and how do I make sure the parallel version returns
> exactly the same answers as the sequential one?

## Why this project

The first four projects are studies. This one is a tool. I wanted to practise writing code
that someone else could import and use, with a clean class interface, a documented modelling
assumption, and a correctness check rather than a claim.

N-1 security is the underlying requirement in both major reliability frameworks. In Europe
it is defined in Commission Regulation (EU) 2017/1485, Article 3: the elements remaining in
operation after a contingency must accommodate the new situation without violating
operational security limits. In North America the planning equivalent is NERC TPL-001.


## 1. The thermal rating problem, and how I handled it

The University of Washington archive is explicit that the line MVA limits in the 118-bus
case were not part of the original 1962 AEP data and were made up. With pandapower's
defaults the base case peaks at 4.5 percent loading, so nothing ever violates and the
screening is vacuous.

I therefore derive ratings as 1.3 times the base case current with a floor, which puts the
base case at 76.92 percent loading and leaves headroom that
contingencies can eat into.

This is a modelling choice, not measured data. I am stating it here rather than in a
footnote because every violation count in this notebook depends on it. A different rating
rule would give different counts. What does not depend on it is the timing comparison,
which is the actual subject of the project.


In [ ]:
import time, json, copy
import numpy as np, pandas as pd
import pandapower as pp, pandapower.networks as pn
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor

OUT = "/tmp/work/out/"
V_MIN, V_MAX, LOAD_MAX = 0.94, 1.06, 100.0
_NET = None

def build_net():
    """IEEE 118-bus with derived thermal ratings.

    The University of Washington archive states the line MVA limits were not part
    of the original 1962 AEP data and were made up. With pandapower's defaults the
    base case peaks at 4.5 percent loading, which makes N-1 screening vacuous.
    Ratings are therefore derived as 1.3x the base case apparent flow with a
    20 MVA floor. This is a modelling choice, not measured data, and is stated
    as such in the write-up.
    """
    net = pn.case118()
    pp.runpp(net, numba=False)
    i_base = net.res_line.i_ka.values
    net.line["max_i_ka"] = np.maximum(1.3 * i_base, 0.05) * net.line.parallel.values
    pp.runpp(net, numba=False)
    assert net.res_line.loading_percent.max() <= 100.0, "base case must be secure"
    return net

net = build_net()
print(f"buses {len(net.bus)}, lines {len(net.line)}, base max loading {net.res_line.loading_percent.max():.2f}%")

## 2. The screening function

Two details matter for making this parallelisable.

The worker function is defined at module level rather than as a method or a closure, because
`ProcessPoolExecutor` has to pickle it to send it to a subprocess. A bound method or a lambda
fails here, and the error message is not obvious the first time you meet it.

The network is passed once per worker through an initialiser rather than once per task.
Sending a copy of the whole network with every one of the 173 tasks would cost more in
serialisation than the power flow costs to solve.


In [ ]:
def _init(net):
    global _NET
    _NET = net

def _screen(idx):
    net = _NET
    net.line.at[idx, "in_service"] = False
    try:
        pp.runpp(net, numba=False)
        r = dict(line=int(idx), converged=True,
            max_loading=float(net.res_line.loading_percent.max()),
            v_min=float(net.res_bus.vm_pu.min()), v_max=float(net.res_bus.vm_pu.max()),
            n_thermal=int((net.res_line.loading_percent > LOAD_MAX).sum()),
            n_voltage=int((net.res_bus.vm_pu < V_MIN).sum() + (net.res_bus.vm_pu > V_MAX).sum()))
    except Exception:
        r = dict(line=int(idx), converged=False, max_loading=np.nan,
                 v_min=np.nan, v_max=np.nan, n_thermal=0, n_voltage=0)
    net.line.at[idx, "in_service"] = True
    return r

## 3. The analyzer class

In [ ]:
class ContingencyAnalyzer:
    """Screen every single line outage on a network, sequentially or in parallel."""
    def __init__(self, net): self.net = net
    def screen(self, parallel=False, workers=None):
        idxs = list(self.net.line.index)
        t0 = time.perf_counter()
        if parallel:
            with ProcessPoolExecutor(max_workers=workers, initializer=_init,
                                     initargs=(self.net,)) as ex:
                recs = list(ex.map(_screen, idxs, chunksize=8))
        else:
            _init(copy.deepcopy(self.net))
            recs = [_screen(i) for i in idxs]
        df = pd.DataFrame(recs); df["n_violations"] = df.n_thermal + df.n_voltage
        return df, time.perf_counter() - t0

## 4. Running the screen

In [ ]:
    net = build_net()
    base = dict(n_bus=len(net.bus), n_line=len(net.line), n_trafo=len(net.trafo),
                n_gen=len(net.gen), n_load=len(net.load),
                total_load_mw=round(float(net.load.p_mw.sum()), 1),
                base_max_loading=round(float(net.res_line.loading_percent.max()), 2),
                base_v_min=round(float(net.res_bus.vm_pu.min()), 4),
                base_v_max=round(float(net.res_bus.vm_pu.max()), 4))
    print("BASE", json.dumps(base), flush=True)

    a = ContingencyAnalyzer(net)
    seq, t_seq = a.screen(parallel=False)
    print(f"sequential {t_seq:.2f}s", flush=True)
    timings = {"1": t_seq}
    par4 = None
    for w in (2, 4):
        df, t = a.screen(parallel=True, workers=w)
        timings[str(w)] = t
        if w == 4: par4 = df
        print(f"workers={w}  {t:.2f}s  speedup {t_seq/t:.2f}x", flush=True)

## 5. Correctness before speed

A faster wrong answer is worthless, so the parallel and sequential results are compared
element by element before any timing is reported.


In [ ]:
    m = seq.merge(par4, on="line", suffixes=("_s", "_p"))
    identical = bool(np.allclose(m.max_loading_s.fillna(-1), m.max_loading_p.fillna(-1))
                     and (m.n_violations_s == m.n_violations_p).all())
    print("identical results:", identical, flush=True)

## 6. Results

| Quantity | Value |
|---|---|
| Contingencies screened | 173 |
| Converged | 173 |
| With thermal violations | 129 |
| With voltage violations | 12 |
| Worst case | line 46 at 619.3 percent |

| Workers | Runtime | Speed-up |
|---|---|---|
| 1 (sequential) | 4.90 s | 1.00x |
| 2 | 2.83 s | 1.73x |
| 4 | 1.72 s | 2.85x |

**Speed-up of 2.85x on 4 cores, a runtime reduction of 64.9 percent,
with results verified identical to the sequential run.**

The scaling is sub-linear, 2.85x rather than 4x. Process startup, pickling the network
to each worker, and collecting results back all cost time that does not shrink with more
cores. On a problem this size the fixed overhead is a visible fraction of the total. It would
matter less on a larger network or a longer contingency list.

**On the security result itself.** 129 of 173 single line
outages cause a thermal violation under my derived ratings. That is a statement about my
rating assumption, not about the 118-bus system, and I would not quote it as a finding.


In [ ]:
    seq = seq.sort_values("max_loading", ascending=False)
    seq.to_csv(OUT+"p05_contingency_118bus.csv", index=False)
    best = min(timings.values())
    summary = dict(base=base, timings={k: round(v,3) for k,v in timings.items()},
        identical_results=identical, cpu_count=4,
        n_contingencies=int(len(seq)), n_converged=int(seq.converged.sum()),
        n_diverged=int((~seq.converged).sum()),
        n_with_violations=int((seq.n_violations>0).sum()),
        n_thermal_cases=int((seq.n_thermal>0).sum()),
        n_voltage_cases=int((seq.n_voltage>0).sum()),
        worst_line=int(seq.iloc[0].line), worst_loading=round(float(seq.iloc[0].max_loading),1),
        speedup=round(t_seq/best,2), runtime_reduction_pct=round(100*(1-best/t_seq),1))
    json.dump(summary, open(OUT+"p05_summary.json","w"), indent=1)
    print(json.dumps(summary, indent=1), flush=True)
    print(seq.head(10)[["line","max_loading","v_min","n_thermal","n_voltage"]].to_string(index=False))

## 7. Figures

In [ ]:
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
    ok = seq[seq.converged]
    ax[0].bar(range(len(ok)), ok.max_loading, width=1.0,
              color=["#C4714A" if v > LOAD_MAX else "#4A7C6F" for v in ok.max_loading])
    ax[0].axhline(LOAD_MAX, ls="--", c="#1A1714", lw=1)
    ax[0].annotate("thermal limit", (len(ok)*0.45, LOAD_MAX*1.04), fontsize=9)
    ax[0].set(xlabel="Contingency, ranked by severity", ylabel="Max line loading [%]",
              title=f"N-1 screening, IEEE 118-bus, {len(ok)} outages")
    ks = sorted(int(k) for k in timings)
    ax[1].plot(ks, [t_seq/timings[str(k)] for k in ks], "o-", color="#4A7C6F", lw=2, label="measured")
    ax[1].plot([1, max(ks)], [1, max(ks)], ls=":", c="#9B948C", label="linear scaling")
    ax[1].set(xlabel="Worker processes", ylabel="Speed-up vs sequential",
              title="Parallel scaling on 4 cores", xticks=ks); ax[1].legend(fontsize=9)
    for x in ax: x.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(OUT+"p05_results.png", dpi=160)

## 8. Limitations

- Line outages only. Generator and transformer contingencies are the obvious extension.
- AC power flow throughout. Real screening tools run a fast DC pass first and only solve AC
  for the cases that look marginal, which is a much bigger speed-up than parallelism.
- Thermal ratings are derived, not real.
- Four cores. The interesting scaling questions start at 32 or more.
- No N-1-1 or common mode outages.

## 9. What I would do next

- Add a DC pre-screen and measure how much of the AC work it removes.
- Extend to generator and transformer outages.
- Rank contingencies by severity index rather than raw loading, which is what planners use.
- Package it properly with tests so it can be pip installed.

## References

1. Christie, R. (1993). "118 Bus Power Flow Test Case." Power Systems Test Case Archive,
   University of Washington. https://labs.ece.uw.edu/pstca/pf118/pg_tca118bus.htm
   Source of the statement that the line MVA limits were not in the original data.
2. Thurner, L. et al. (2018). "pandapower: An Open-Source Python Tool for Convenient
   Modeling, Analysis, and Optimization of Electric Power Systems." *IEEE Transactions on
   Power Systems*, 33(6), 6510 to 6521. DOI: 10.1109/TPWRS.2018.2829021
3. Commission Regulation (EU) 2017/1485 of 2 August 2017 establishing a guideline on
   electricity transmission system operation. *Official Journal of the European Union.*
   https://eur-lex.europa.eu/eli/reg/2017/1485/oj/eng
   Article 3 defines the (N-1) criterion.
4. North American Electric Reliability Corporation. "TPL-001-5.1: Transmission System
   Planning Performance Requirements." Reliability Standard.
   https://www.nerc.com/globalassets/standards/reliability-standards/tpl/tpl-001-5.1.pdf
5. Illinois Center for a Smarter Electric Grid. "IEEE 118-Bus System."
   https://icseg.iti.illinois.edu/ieee-118-bus-system/
